In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorWithPadding
import numpy as np
import torch
from torch import nn
import numpy as np

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

c:\Users\xadic\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('sentiment_binary_full.csv')
df

,text,label
0,super👍👍,1
1,çox razıyam,1
2,Əlaa,1
3,rəngi bir tık tünd götürmədiyimə peşmanam 🥲 Be...,1
4,normaldi qiymwtine gore,1
...,...,...
2428,maşın yağının qoxusu gelir,0
2429,10 günə 5 kq arıqladım. Superdi👍,1
2430,1 qutu ilə 6 kq arıqladımm😍,1
2431,Super,1


In [3]:
X = df['text']
y = df['label']

In [4]:
X_train , X_temp , y_train, y_temp = train_test_split(
    X,
    y,
    test_size = 0.3,
    random_state = 42,
    stratify = y
)

X_test , X_val , y_test, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size = 0.5,
    random_state = 42,
    stratify = y_temp

)

In [5]:
train_df = X_train.to_frame()
train_df['label'] = y_train.values

test_df = X_test.to_frame()
test_df['label'] = y_test.values

val_df = X_val.to_frame()
val_df['label'] = y_val.values

In [6]:
train_df.label.value_counts()

label
1    1557
0     146
Name: count, dtype: int64

In [19]:
neg = train_df[train_df["label"] == 0]
pos = train_df[train_df["label"] == 1]

neg_oversampled = neg.sample(
    n=len(pos),
    replace=True,
    random_state=42
)

train_balanced = pd.concat([pos, neg_oversampled])
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [20]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_balanced.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_dataset



Map: 100%|██████████| 365/365 [00:00<00:00, 26071.54 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 3114
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 365
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 365
    })
})

In [21]:
id2label = {
    0: 'negative',
    1: 'positive'
}

label2id = {
    'negative': 0,
    'positive': 1
}

In [22]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 2,
    id2label = id2label,
    label2id = label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1669.73it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [26]:
training_args = TrainingArguments(
    output_dir="./sentiment_xlm_roberta",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=5,

    weight_decay=0.01,
    warmup_ratio=0.1,

    logging_strategy="steps",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none"
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [27]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    data_collator=data_collator,
    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

In [28]:
trainer.train()

c:\Users\xadic\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.355414,0.542068,0.857534,0.680048,0.892892,0.721701
2,0.258161,0.761231,0.945205,0.834100,0.794476,0.812705
3,0.070273,0.597770,0.945205,0.819779,0.838372,0.828735
4,0.070373,0.767504,0.947945,0.840822,0.810605,0.824825
5,0.025276,0.835406,0.945205,0.828459,0.809108,0.818408


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.17s/it]
c:\Users\xadic\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.59s/it]
c:\Users\xadic\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.85s/it]
c:\Users\xadic\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█

TrainOutput(global_step=1950, training_loss=0.1628782025973002, metrics={'train_runtime': 6018.188, 'train_samples_per_second': 2.587, 'train_steps_per_second': 0.324, 'total_flos': 324583459001160.0, 'train_loss': 0.1628782025973002, 'epoch': 5.0})

In [29]:
preds_output = trainer.predict(tokenized_dataset['test'])
y_true = preds_output.label_ids
y_pred = preds_output.predictions.argmax(axis=1)

c:\Users\xadic\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [30]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

[[ 22   9]
 [ 11 323]]
              precision    recall  f1-score   support

           0       0.67      0.71      0.69        31
           1       0.97      0.97      0.97       334

    accuracy                           0.95       365
   macro avg       0.82      0.84      0.83       365
weighted avg       0.95      0.95      0.95       365

